# AI Stylist (Ari) - Simple Test Notebook

This notebook provides a simple interface to interact with the AI Stylist and analyze response times.

## Setup

First, let's install the necessary packages and set up the environment.

In [ ]:
# Install required packages if needed
!pip install ipywidgets requests matplotlib pandas

In [ ]:
import os
import time
import json
import requests
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

# Set API URL (assuming the stylist service is running)
API_URL = "http://localhost:5000"

## Check Service Availability

Let's check if the AI Stylist service is running.

In [ ]:
def check_service():
    try:
        response = requests.get(f"{API_URL}/health", timeout=5)
        if response.status_code == 200:
            print(f"✅ Service is running! Status: {response.json().get('status')}")
            return True
        else:
            print(f"⚠️ Service returned unexpected status code: {response.status_code}")
            return False
    except requests.exceptions.RequestException as e:
        print(f"❌ Service connection failed: {e}")
        print("Make sure the AI Stylist service is running by executing 'python stylist_service.py'")
        return False

SERVICE_AVAILABLE = check_service()

## Chat with AI Stylist

This section provides a simple chat interface to interact with the AI Stylist.

In [ ]:
def send_message(message, session_id=None):
    """Send a message to the AI Stylist service"""
    try:
        data = {"message": message}
        if session_id:
            data["session_id"] = session_id
        else:
            data["user_id"] = "notebook_user"
            
        start_time = time.time()
        response = requests.post(f"{API_URL}/chat", json=data, timeout=30)
        response_time = time.time() - start_time
        
        if response.status_code == 200:
            result = response.json()
            return {
                "success": result.get("success", True),
                "response": result.get("response", ""),
                "session_id": result.get("session_id"),
                "response_time": response_time
            }
        else:
            return {
                "success": False,
                "response": f"Error: HTTP {response.status_code}",
                "response_time": response_time
            }
    except Exception as e:
        return {
            "success": False,
            "response": f"Error: {str(e)}",
            "response_time": 0
        }

In [ ]:
# Chat history and session tracking
chat_history = []
current_session_id = None
response_times = []

# Create chat widgets
message_input = widgets.Textarea(
    placeholder='Type your message here...',
    layout=widgets.Layout(width='100%', height='80px')
)

send_button = widgets.Button(
    description='Send',
    button_style='primary',
    icon='paper-plane'
)

new_session_button = widgets.Button(
    description='New Session',
    button_style='warning',
    icon='plus'
)

output_area = widgets.Output(
    layout=widgets.Layout(width='100%', height='400px', border='1px solid #ddd', overflow_y='auto')
)

stats_area = widgets.Output(
    layout=widgets.Layout(width='100%', height='100px', border='1px solid #ddd')
)

# Custom styling for chat messages
chat_style = """
<style>
.chat-container {
    display: flex;
    flex-direction: column;
    gap: 10px;
    padding: 10px;
    font-family: Arial, sans-serif;
}
.user-message {
    align-self: flex-end;
    background-color: #DCF8C6;
    padding: 10px 15px;
    border-radius: 15px 0px 15px 15px;
    max-width: 80%;
    margin-left: 20%;
    box-shadow: 0 1px 0.5px rgba(0,0,0,0.13);
}
.stylist-message {
    align-self: flex-start;
    background-color: #E3E3E3;
    padding: 10px 15px;
    border-radius: 0px 15px 15px 15px;
    max-width: 80%;
    margin-right: 20%;
    box-shadow: 0 1px 0.5px rgba(0,0,0,0.13);
}
.message-info {
    font-size: 0.8em;
    color: #666;
    margin-top: 5px;
}
.message-container {
    margin-bottom: 10px;
}
</style>
"""

# Function to update the chat display
def update_chat_display():
    with output_area:
        clear_output()
        display(HTML(chat_style + "<div class='chat-container'>" +
               "".join([f"<div class='message-container'>"
                         f"<div class='{msg['sender']}-message'>{msg['content']}</div>"
                         f"<div class='message-info'>{msg['timestamp']} - {msg['response_time']:.2f}s</div>"
                         f"</div>" for msg in chat_history]) +
               "</div>"))
        
    with stats_area:
        clear_output()
        if response_times:
            avg_time = sum(response_times) / len(response_times)
            max_time = max(response_times)
            min_time = min(response_times)
            print(f"Session ID: {current_session_id}")
            print(f"Messages: {len(chat_history) // 2}")
            print(f"Response times - Avg: {avg_time:.2f}s, Min: {min_time:.2f}s, Max: {max_time:.2f}s")

# Handler for the send button
def on_send_button_clicked(b):
    global current_session_id
    message = message_input.value.strip()
    if not message:
        return
    
    # Add user message to chat history
    timestamp = time.strftime("%H:%M:%S")
    chat_history.append({
        "sender": "user",
        "content": message,
        "timestamp": timestamp,
        "response_time": 0
    })
    update_chat_display()
    
    # Clear input
    message_input.value = ""
    
    # Send message to stylist
    result = send_message(message, session_id=current_session_id)
    
    # Track session ID
    if result.get("session_id"):
        current_session_id = result.get("session_id")
    
    # Track response time
    response_time = result.get("response_time", 0)
    response_times.append(response_time)
    
    # Add stylist response to chat history
    timestamp = time.strftime("%H:%M:%S")
    chat_history.append({
        "sender": "stylist",
        "content": result.get("response", "Error: No response"),
        "timestamp": timestamp,
        "response_time": response_time
    })
    update_chat_display()

# Handler for the new session button
def on_new_session_button_clicked(b):
    global current_session_id, chat_history, response_times
    
    # Clear chat history and response times
    chat_history = []
    response_times = []
    current_session_id = None
    
    # Update display
    update_chat_display()
    with output_area:
        print("New session will be created on first message")

# Connect handlers to buttons
send_button.on_click(on_send_button_clicked)
new_session_button.on_click(on_new_session_button_clicked)

# Create the chat UI
chat_ui = widgets.VBox([
    widgets.HTML("<h2>AI Stylist Chat</h2>"),
    output_area,
    widgets.HBox([message_input, send_button, new_session_button]),
    stats_area
])

# Display the chat UI
display(chat_ui)

# Initialize with a new session
on_new_session_button_clicked(None)

## Response Time Analysis

Analyze and visualize the response times from your conversation.

In [ ]:
def plot_response_times():
    """Plot the response times from the current conversation"""
    if not response_times:
        print("No response times to plot. Please chat with the stylist first.")
        return
    
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, len(response_times) + 1), response_times, marker='o', linestyle='-', color='blue')
    plt.axhline(y=sum(response_times) / len(response_times), color='r', linestyle='--', label='Average')
    
    plt.xlabel('Message Number')
    plt.ylabel('Response Time (seconds)')
    plt.title('AI Stylist Response Times')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend()
    plt.tight_layout()
    plt.show()
    
    # Print statistics
    print(f"Response Time Statistics:")
    print(f"Average: {sum(response_times) / len(response_times):.2f}s")
    print(f"Minimum: {min(response_times):.2f}s")
    print(f"Maximum: {max(response_times):.2f}s")
    print(f"Total conversation time: {sum(response_times):.2f}s")
    
    # Create a DataFrame for more detailed analysis
    df = pd.DataFrame({
        'Message': range(1, len(response_times) + 1),
        'Response Time (s)': response_times
    })
    
    # Display the data
    display(df)

In [ ]:
# Plot the response times from your conversation
plot_response_times()

## Sample Fashion Queries

Here are some sample queries you can copy-paste to test the AI Stylist:

In [ ]:
print("Sample Fashion Queries:")
print("1. I need something to wear to a summer wedding next month.")
print("2. Can you recommend a business casual outfit for a job interview?")
print("3. I'm looking for comfortable yet stylish work-from-home outfits.")
print("4. What are some trendy outfits for fall?")
print("5. I need a stylish outfit for a date night.")
print("6. What should I wear for a beach vacation?")
print("7. Can you suggest some accessories to go with a black dress?")
print("8. I need outfit ideas for a winter formal event.")
print("9. I need help finding a good outfit for a family photo shoot.")
print("10. I'm looking for a versatile dress that works for multiple occasions.")